In [1]:
from datetime import datetime
import pandas as pd
import numpy as np
import os 

## import helper 

In [2]:
platformID = 'FBE'

In [3]:
import sys
from pathlib import Path

try:
    # Works in Python scripts
    helper_path = Path(__file__).resolve().parent.parent / "helper"
except NameError:
    # Works in Jupyter notebooks
    helper_path = Path().resolve().parent / "helper"

sys.path.insert(0, str(helper_path))

# Now import your modules 
from functions import execute_sql_query
import test_functions

from config import gam_info

In [4]:
# week 
week_cols = ['w/c']
week_tester = pd.read_excel(f"../../{gam_info['lookup_file']}", sheet_name='GAM Period')

# social media accounts
channel_cols=['Channel ID']
dtype_dict = {'Channel ID': 'str',
              'Linked FB Account': 'str'}
socialmedia_accounts = pd.read_excel(f"../../{gam_info['lookup_file']}", dtype=dtype_dict,
                                     sheet_name='Social Media Accounts new')

# socialmedia_accounts = socialmedia_accounts[socialmedia_accounts['Year'] == gam_info['file_timeinfo']]
socialmedia_accounts = socialmedia_accounts[socialmedia_accounts['PlatformID'] == platformID]
socialmedia_accounts = socialmedia_accounts[socialmedia_accounts['Status'] == 'active']
socialmedia_accounts['Channel ID'] = socialmedia_accounts['Channel ID'].dropna().apply(lambda x: str(int(x)))
socialmedia_accounts['Channel ID'] = platformID + socialmedia_accounts['Channel ID']
channel_ids = socialmedia_accounts['Channel ID'].unique().tolist()
formatted_channel_ids = ', '.join(f"'{channel_id}'" for channel_id in channel_ids)

### RUN TESTS
test_functions.test_lookup_files(week_tester, ['w/c'], [f"{platformID}_1e_0", f"{platformID}_1e_1", f"{platformID}_1e_2"], 
                                 test_step = "lookup files - ensuring week tester is correct")

test_functions.test_lookup_files(socialmedia_accounts, ['Channel ID'], [f"{platformID}_1e_3", f"{platformID}_1e_4", f"{platformID}_1e_5"], 
                                 test_step = "lookup files - ensuring social media accounts is correct")


✅ Test FBE_1e_0 passed: lookup DataFrame is not empty.
...updating logbook...

✅ Test FBE_1e_1 passed: No combinations occurs more than once a week.
...updating logbook...

✅ Test FBE_1e_2 passed: No missing values in lookup.
...updating logbook...

✅ Test FBE_1e_3 passed: lookup DataFrame is not empty.
...updating logbook...

✅ Test FBE_1e_4 passed: No combinations occurs more than once a week.
...updating logbook...

✅ Test FBE_1e_5 passed: No missing values in lookup.
...updating logbook...



# engagements 

In [5]:
sql_query = f"""
    SELECT
        week_commencing,
        page_id,
        CASE
            WHEN (AVG(engagements)/AVG(post_engagements_to_page_consumptions))/AVG(avg_engagements_per_user) 
                 > AVG(video_views)/AVG(page_video_views_to_10s_unique_viewer)
            THEN ((AVG(engagements)/AVG(post_engagements_to_page_consumptions))/AVG(avg_engagements_per_user)) 
                 + (AVG(video_views)/AVG(page_video_views_to_10s_unique_viewer))*0.04827
            ELSE (AVG(video_views)/AVG(page_video_views_to_10s_unique_viewer)) 
                 + ((AVG(engagements)/AVG(post_engagements_to_page_consumptions))/AVG(avg_engagements_per_user))*0.04822
        END AS engaged_reach
    FROM 
        redshiftdb.central_insights.adverity_social_facebook_by_page AS p
    RIGHT JOIN
        world_service_audiences_insights.social_media_lookup_fb AS l
        ON p.page_id = l.fb_page_id
    WHERE 
        period = 'week'
    AND
        week_commencing BETWEEN '{gam_info['w/c_start']}' AND '{gam_info['w/c_end']}'
    GROUP BY 
        week_commencing, page_id
        ;
"""

file = f"../data/raw/{platformID}/{gam_info['file_timeinfo']}_{platformID}_engagements_redshift_extract.csv"

# TODO make this more fail safe - e.g. I want to run this and say if it should query redshift or not
try: 
    df = execute_sql_query(sql_query)
    df['page_id'] = platformID+df['page_id'].astype(str)
    df.to_csv(file, index=False, na_rep='')
except:
    print("no redshift connection using last time queried")
    
facebook_engagements_raw = pd.read_csv(file, keep_default_na=False)
facebook_engagements_raw['page_id'] = facebook_engagements_raw['page_id'].astype(str)
facebook_engagements_raw['week_commencing'] = pd.to_datetime(facebook_engagements_raw['week_commencing'])
facebook_engagements_raw = facebook_engagements_raw.rename(columns={'page_id': 'Channel ID', 
                                                                    'week_commencing': 'w/c'})
print(facebook_engagements_raw.shape)


no redshift connection using last time queried
(3754, 3)


In [6]:

### RUN TESTS
# missing page_ids
test_functions.test_filter_elements_returned(facebook_engagements_raw, 
                                             channel_ids, 
                                             'Channel ID', 
                                             f"{platformID}_1e_6",
                                             "Check that all page IDs are found in SQL")

# missing weeks per page_id
test_functions.test_weeks_presence_per_account(key='w/c',
                                               main_data=facebook_engagements_raw,
                                               week_lookup=week_tester[['w/c']],
                                               channel_lookup=socialmedia_accounts[['Channel ID', 'Start', 'End']],
                                               test_number=f"{platformID}_1e_7",
                                               test_step="Check all weeks present for each account")

# missing values per week / page id 
test_functions.test_non_null_and_positive(facebook_engagements_raw, 
                           numeric_columns=['engaged_reach'], 
                           test_number=f"{platformID}_1e_8",
                           test_step='Check no missing values in engaged_reach column from redshift returned')

# test for duplicate entries 
test_functions.test_duplicates(facebook_engagements_raw, 
                               ['Channel ID', 'w/c'], 
                               test_number=f"{platformID}_1e_9",
                               test_step='Check no duplicates from redshift returned')

...testing Channel ID...
✅ Test FBE_1e_6 passed: everything found!
...updating logbook...

❌ Missing weeks from Start onward:
          Start        End           Channel ID        w/c
1561 2025-04-02 2025-08-19   FBE171824429536304 2025-04-28
1563 2025-04-02 2025-08-19   FBE171824429536304 2025-05-12
1565 2025-04-02 2025-08-19   FBE171824429536304 2025-05-26
37   2020-04-01 2025-12-23   FBE100163990128209 2025-12-15
2582 2020-04-01 2025-12-23   FBE303095973088848 2025-12-15
...         ...        ...                  ...        ...
1101 2020-04-01 2025-12-23      FBE146214266618 2025-12-15
1063 2020-04-01 2025-12-23   FBE143048895744759 2025-12-15
987  2020-04-01 2025-12-23  FBE1385243528221504 2025-12-15
1367 2020-04-01 2025-12-23   FBE160894643929209 2025-12-15
3836 2020-04-01 2025-12-23   FBE948946275170651 2025-12-15

[103 rows x 4 columns]

Summary of missing weeks per Channel ID:
             Channel ID  missing_week_count
43   FBE171824429536304                   3
0    FBE1001

/Users/brunsd01/BBC Dropbox/Domi Bruns/BBC - InfoLab - Lumen 2020/digiGAM_ytd/digiGAM/helper/test_functions.py:543: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  week_lookup_test_data[key] = pd.to_datetime(week_lookup_test_data[key], errors='coerce', dayfirst=True)


## processing engagements

In [7]:
# needs to be right because we want to make sure that all the accounts in lookup are there 
facebook_engagements = facebook_engagements_raw.merge(socialmedia_accounts[['Channel ID', 'ServiceID']], 
                                                      on='Channel ID', how='right', indicator=True)
test_functions.test_inner_join(facebook_engagements_raw, socialmedia_accounts, 
                               ['Channel ID'], 
                               f"{platformID}_1e_10", 
                               test_step='checking social media accounts in lookup, adding service',
                               focus='right')


✅ Inner join test FBE_1e_10 successful: No issues found.
...updating logbook...



In [8]:
file_path = f"../data/processed/{platformID}"
os.makedirs(file_path, exist_ok=True)

cols = ['Channel ID', 'ServiceID', 'w/c', 'engaged_reach']
facebook_engagements[cols].to_csv(f"{file_path}/{gam_info['file_timeinfo']}_{platformID}_REDSHIFT.csv",
                                       index=None)
